# Phase 09 — Candidate Job Reranking

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Align recommendation training with backend-owned candidate retrieval instead of static job-index recommendation.

This notebook defines the candidate reranking input contract, model-core output contract, constraint validation, ranking evaluation policy, and backend staleness boundary. It writes `reports/phase_09_candidate_reranking.json` as the machine-readable review artifact.

## Purpose
Document and verify Phase 09 — Candidate Job Reranking in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 09.candidate.reranking notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 09 — Candidate Job Reranking.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Contract boundary

Backend candidate retrieval is the source of truth for which jobs may be ranked. The model/core receives a bounded `jobCandidates` list and may only score, rank, and explain those supplied candidates.

The model must not search a static training artifact for final recommendations, invent job IDs, hydrate job detail, decide visibility, or override backend freshness and availability rules.

## Shared setup

### Purpose
Load generated API schema evidence and prior ranking blockers, then prepare constants used by every reranking step.

### Required input
Repository root with `TODOS.md`, `references/docs/generated/openapi.json`, `reports/phase_05_baseline_evaluation.json`, and `reports/phase_06_jobfit_training_experiments.json`.

### Action
Read public job recommendation schemas, extract match-level enum values and public response caps, and collect inherited training blockers.

### Expected output
Reusable constants for candidate recommendation contracts, public response limits, and known readiness blockers.

### Verification
Fail fast if required files are missing. Confirm public recommendation schemas expose bounded scores and stable match-level values.

In [13]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / "reports"
OPENAPI_PATH = ROOT / "references" / "docs" / "generated" / "openapi.json"
PHASE5_PATH = REPORTS / "phase_05_baseline_evaluation.json"
PHASE6_PATH = REPORTS / "phase_06_jobfit_training_experiments.json"
REPORT_PATH = REPORTS / "phase_09_candidate_reranking.json"


def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path.relative_to(ROOT)}")
    return json.loads(path.read_text())


openapi = read_json(OPENAPI_PATH)
phase5 = read_json(PHASE5_PATH)
phase6 = read_json(PHASE6_PATH)
components = openapi["components"]["schemas"]
job_rec_item = components["JobRecommendationItem"]
cv_rec_item = components["CvAnalysisJobRecommendation"]
match_levels = job_rec_item["properties"]["matchLevel"]["enum"]
public_cv_max_items = components["CvAnalysis"]["properties"]["analysisResult"]["properties"]["jobRecommendations"].get("maxItems")

setup_summary = {
    "match_levels": match_levels,
    "public_cv_max_items": public_cv_max_items,
    "standalone_item_required": job_rec_item["required"],
    "cv_item_required": cv_rec_item["required"],
    "inherited_ranking_blockers": sorted(set(phase5.get("blocked_until_later_phases", []) + phase6.get("blocked_until_later_phases", []))),
}
setup_summary


{'match_levels': ['strong', 'good', 'stretch'],
 'public_cv_max_items': 5,
 'standalone_item_required': ['job',
  'matchScore',
  'matchLevel',
  'reasons',
  'matchedSkills',
  'missingSkills',
  'nextSteps',
  'isBookmarked',
  'hasApplied'],
 'cv_item_required': ['jobId',
  'title',
  'companyName',
  'matchScore',
  'reason',
  'nextStep'],
 'inherited_ranking_blockers': ['Complex JobFitAlignment training must wait for a materialized balanced pair dataset with high-fit validation/test coverage.',
  'Phase 10 calibration must wait for stable label distribution and model outputs.',
  'Phase 6 complex job-fit training must wait for balanced pairs, high-fit coverage, and stronger labels.',
  'Phase 9 recommendation ranking must wait for backend candidate-set labels or realistic candidate sets.',
  'Production score semantics must wait for Phase 10 calibration.',
  'Promotion must wait for human validation labels or equivalent trusted validation evidence.',
  'Ranking-objective promotio

## Step 9.1 — Input contract

### Purpose
Document that backend provides `jobCandidates` and the model scores only those candidates.

### Required input
Use backend-prepared `requestId`, `inputVersion`, `talentProfile`, `rankingPolicy`, and `jobCandidates`. Candidate rows must include stable `jobId` and enough text/skill/context evidence for ranking.

### Action
Define the candidate-set input contract, owner boundary, disallowed inference inputs, and training dataset requirements for grouped reranking evaluation.

### Expected output
A machine-readable input contract requiring backend-owned candidate sets and blocking static full-index recommendation output.

### Verification
Confirm `jobCandidates` is required, backend remains owner of candidate retrieval, and inference features exclude labels, stale final job detail, and frontend-trusted raw profile data.

## Step 9.2 — Output contract

### Purpose
Define output fields: `jobId`, `matchScore`, `matchLevel`, `matchedSkills`, `missingSkills`, `rankingSignals`, and optional `nextSteps`.

### Required input
Use the validated candidate set, normalized talent evidence, prior job-fit feature contracts, and generated public API schemas.

### Action
Define model-owned fields, wrapper-owned fields, stable match-level values, score range, deterministic ordering, and public API mapping.

### Expected output
A reviewable output contract where every `jobId` comes from input candidates and every explanation is grounded in skills or ranking signals.

### Verification
Confirm model-core output excludes hydrated job details, uses integer `0-100` scores, and keeps wrapper prose separate from model evidence.

## Step 9.3 — Constraint validation

### Purpose
Define checks for no unknown `jobId`, no duplicate `jobId`, and every output `jobId` belonging to input candidates.

### Required input
Use the exact input candidate IDs, model recommendations, score values, and ranking policy limits.

### Action
Define hard-fail validation checks, soft diagnostics, and zero-tolerance acceptance thresholds for membership, uniqueness, score validity, and response count.

### Expected output
A validation policy that rejects invented jobs and makes output constraints measurable in reports and serving validators.

### Verification
Confirm unknown-job, duplicate-job, membership, invalid-score, and max-recommendation violations are hard failures with zero accepted rate.

## Step 9.4 — Ranking evaluation

### Purpose
Define NDCG@5, NDCG@10, MAP@10, and comparison against the best ranking baseline.

### Required input
Use grouped candidate sets with graded or binary relevance labels, plus backend retrieval order, skill-overlap, cosine-only, and current best baseline scores.

### Action
Specify required metrics, baseline comparisons, slice analysis, and readiness decision for reranking promotion.

### Expected output
A ranking evaluation plan that can prove top-k improvement without hiding constraint failures.

### Verification
Confirm ranking metrics are computed per candidate set, averaged across validation/test groups, and reported with constraint violation rates and slice metrics.

## Step 9.5 — Staleness boundary

### Purpose
Document that backend remains owner of job detail, visibility, availability, and hydration.

### Required input
Use backend candidate metadata such as `postedAt`, `sourceUpdatedAt`, visibility status, availability status, and duplicate-source policy when provided.

### Action
Separate backend ownership from allowed model diagnostics. Define what the model must never do with stale artifacts or inactive jobs.

### Expected output
A staleness and hydration boundary that prevents the model from returning stale, private, inactive, or non-candidate jobs.

### Verification
Confirm backend owns candidate retrieval, availability, visibility, source freshness policy, company/location display, bookmark/application state, and final hydration.

## Report generation

### Purpose
Write the complete candidate reranking design artifact.

### Required input
Shared setup constants, generated schema evidence, and prior readiness blockers.

### Action
Build `reports/phase_09_candidate_reranking.json` with input contract, output contract, validation checks, ranking evaluation, staleness boundary, blockers, and acceptance status.

### Expected output
`reports/phase_09_candidate_reranking.json`.

### Verification
The report must satisfy all acceptance criteria and include required fields, hard constraints, and metrics.

In [14]:
report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_references": [
        "TODOS.md",
        "GAP_MODEL_TRAINING.md#GAP-06",
        "references/docs/integrations/model-api.md#Job Recommendation Payload Contract",
        "references/docs/modules/ai-job-recommendations.md#Current Contract",
        "references/docs/generated/openapi.json",
        "reports/phase_05_baseline_evaluation.json",
        "reports/phase_06_jobfit_training_experiments.json",
    ],
    "input_contract": {
        "contract_id": "candidate-reranking-input-v1",
        "owner_boundary": {
            "backend_owns": [
                "user authentication and authorization",
                "candidate retrieval from live backend data",
                "job visibility and availability filtering",
                "deduplication policy before model call when configured",
                "job detail hydration after model output",
            ],
            "model_owns": [
                "score supplied candidates only",
                "rank supplied candidates only",
                "emit grounded skill and ranking signals from candidate evidence",
            ],
        },
        "required_payload_fields": [
            {"field": "requestId", "type": "string", "rule": "Required trace id; never used as predictive feature."},
            {"field": "inputVersion", "type": "string", "rule": "Must identify candidate reranking input schema version, e.g. job-recommendations-v1."},
            {"field": "talentProfile", "type": "object", "rule": "Backend-prepared normalized profile/CV/preference evidence only; no frontend-trusted raw profile object."},
            {"field": "rankingPolicy", "type": "object", "rule": "Backend-supplied limits and constraints; model must obey maxRecommendations, requireCandidateJobIds, and deduplicateByJobId."},
            {"field": "jobCandidates", "type": "array[object]", "rule": "Required backend candidate set. Empty set returns empty recommendations with model metadata, not static artifact jobs."},
        ],
        "disallowed_inputs": [
            "static full job_index ranking output as final recommendations",
            "inactive job ids not supplied by backend",
            "backend hydration fields not needed for scoring",
            "bookmarked/applied flags as ranking features unless explicit product policy approves them",
            "manual validation labels or target relevance as inference-time features",
        ],
        "training_dataset_requirements": [
            "candidate_set_id groups each candidate list for one profile/CV request",
            "job_id is unique within candidate_set_id",
            "candidate_relevance_label or weak relevance label exists for evaluation",
            "split isolation prevents same profile request leaking across train/validation/test",
            "candidate generation source and timestamp are recorded for stale-candidate diagnostics",
        ],
    },
    "output_contract": {
        "contract_id": "candidate-reranking-output-v1",
        "recommendations_shape": {
            "type": "array[object]",
            "ordering": "descending matchScore with deterministic tie break by input order or stable jobId",
            "max_items": "min(rankingPolicy.maxRecommendations, backend public response cap)",
            "public_cv_response_cap": public_cv_max_items,
        },
        "model_owned_fields": [
            {"field": "jobId", "type": "string", "rule": "Must equal one supplied jobCandidates[].jobId; null is not allowed in model-core reranking output."},
            {"field": "matchScore", "type": "integer", "range": "0-100", "rule": "Rounded, bounded score from ranking/scoring model; production semantics wait for calibration evidence."},
            {"field": "matchLevel", "type": "enum", "range": match_levels, "rule": "Derived from calibrated score bands after calibration; before calibration use prototype thresholds only in reports."},
            {"field": "matchedSkills", "type": "array[string]", "rule": "Intersection of normalized candidate skills and job required skills; no invented or proficiency-inferred skills."},
            {"field": "missingSkills", "type": "array[string]", "rule": "Required job skills absent from normalized candidate evidence; exclude optional skills unless policy marks them required."},
            {"field": "rankingSignals", "type": "array[string]", "range": ["skill_overlap", "semantic_similarity", "role_match", "experience_match", "requirement_coverage", "location_preference_match", "work_type_match", "freshness_penalty_applied", "low_evidence_confidence"], "rule": "Stable explainability keys only; wrapper may convert to prose."},
            {"field": "nextSteps", "type": "array[string] | optional", "rule": "Optional model-safe suggestions grounded in missingSkills/rankingSignals; wrapper may own final copy."},
        ],
        "wrapper_owned_fields": ["public reason prose", "single nextStep string for CV Analyzer public response", "job title/company/location/workType/experienceLevel hydration", "isBookmarked", "hasApplied"],
    },
    "constraint_validation": {
        "hard_fail_checks": [
            {"check": "candidate_set_required", "rule": "If rankingPolicy.requireCandidateJobIds is true, jobCandidates must be present before scoring.", "failure_action": "Return validation error to backend; do not fallback to static artifact jobs."},
            {"check": "no_unknown_job_id", "rule": "set(output.jobId) must be subset of set(input.jobCandidates.jobId).", "failure_action": "Reject output and log contract violation."},
            {"check": "no_duplicate_job_id", "rule": "Each output jobId appears at most once.", "failure_action": "Reject output or deterministic dedupe before backend response, depending on serving policy."},
            {"check": "score_range", "rule": "Every matchScore is integer 0-100 and finite before serialization.", "failure_action": "Reject invalid row or whole response; do not clamp silently in production."},
            {"check": "max_recommendations", "rule": "Output count must not exceed rankingPolicy.maxRecommendations or public API cap.", "failure_action": "Trim only after sorting if policy allows; otherwise reject."},
        ],
        "acceptance_thresholds": {"unknown_job_id_rate": 0.0, "duplicate_output_job_id_rate": 0.0, "output_membership_violation_rate": 0.0, "invalid_score_rate": 0.0},
    },
    "ranking_evaluation": {
        "required_metrics": [
            {"metric": "NDCG@5", "purpose": "Measure top-five ordering quality for public CV Analyzer cap."},
            {"metric": "NDCG@10", "purpose": "Measure broader candidate ordering before public cap or standalone internal response."},
            {"metric": "MAP@10", "purpose": "Measure precision of relevant jobs in top ten when relevance is binary or thresholded."},
            {"metric": "constraint_violation_rates", "purpose": "Prove ranking never invents or duplicates jobs."},
        ],
        "baselines": ["backend_candidate_order", "skill_overlap_only", "cosine_only", "best_phase_5_or_phase_6_ranking_baseline"],
        "slice_analysis": ["target_role_or_role_family", "language", "experience_band", "candidate_set_size_bucket", "work_type", "location_preference_presence", "source_freshness_bucket"],
        "current_readiness_decision": {"status": "design_complete_training_blocked", "reason": "Recommendation ranking must wait for backend candidate-set labels or realistic candidate sets."},
    },
    "staleness_boundary": {
        "backend_remains_owner_of": ["candidate retrieval", "job visibility", "job availability/open/closed status", "source freshness policy", "duplicate source collapse", "company and location display fields", "bookmark/application state", "final job detail hydration"],
        "model_must_not_do": ["return jobs outside jobCandidates", "hydrate stale job details from training artifacts", "decide whether inactive/private jobs may be shown", "override backend deduplication or visibility rules"],
    },
    "blocked_until_later_phases": [
        "Training promotion requires backend-like candidate sets with relevance labels or approved weak labels.",
        "Production matchLevel and matchScore semantics require calibration evidence.",
        "Backend response validator must enforce candidate membership, uniqueness, score range, and max item constraints.",
        "Wrapper copy for reasons and next steps must remain grounded in rankingSignals, matchedSkills, and missingSkills.",
    ],
    "acceptance": {
        "recommendation_flow_uses_backend_candidate_sets": True,
        "output_never_invents_jobs": True,
        "ranking_metrics_and_constraints_are_defined": True,
    },
}

REPORTS.mkdir(exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
report["acceptance"]


{'recommendation_flow_uses_backend_candidate_sets': True,
 'output_never_invents_jobs': True,
 'ranking_metrics_and_constraints_are_defined': True}

## Acceptance criteria

- [x] Recommendation flow uses backend candidate sets.
- [x] Output never invents jobs.
- [x] Ranking metrics and constraints are defined.

## Verification

### Purpose
Confirm the saved report satisfies the candidate reranking contract.

### Required input
`reports/phase_09_candidate_reranking.json`.

### Action
Read the report and assert required model fields, hard constraints, metrics, and acceptance criteria.

### Expected output
A compact verification summary with report path, field counts, constraint counts, metric counts, and acceptance flags.

### Verification
All assertions pass.

In [15]:
saved_report = read_json(REPORT_PATH)
model_fields = {field["field"] for field in saved_report["output_contract"]["model_owned_fields"]}
hard_checks = {check["check"] for check in saved_report["constraint_validation"]["hard_fail_checks"]}
metrics = {metric["metric"] for metric in saved_report["ranking_evaluation"]["required_metrics"]}

assert saved_report["acceptance"]["recommendation_flow_uses_backend_candidate_sets"] is True
assert saved_report["acceptance"]["output_never_invents_jobs"] is True
assert saved_report["acceptance"]["ranking_metrics_and_constraints_are_defined"] is True
assert {"jobId", "matchScore", "matchLevel", "matchedSkills", "missingSkills", "rankingSignals"}.issubset(model_fields)
assert {"no_unknown_job_id", "no_duplicate_job_id", "score_range", "candidate_set_required"}.issubset(hard_checks)
assert {"NDCG@5", "NDCG@10", "MAP@10"}.issubset(metrics)

verification_summary = {
    "report_path": str(REPORT_PATH.relative_to(ROOT)),
    "model_owned_field_count": len(model_fields),
    "hard_constraint_count": len(hard_checks),
    "metric_count": len(metrics),
    "acceptance": saved_report["acceptance"],
}
verification_summary


{'report_path': 'reports/phase_09_candidate_reranking.json',
 'model_owned_field_count': 7,
 'hard_constraint_count': 5,
 'metric_count': 4,
 'acceptance': {'output_never_invents_jobs': True,
  'ranking_metrics_and_constraints_are_defined': True,
  'recommendation_flow_uses_backend_candidate_sets': True}}

## Phase notes

- Training promotion remains blocked until backend-like candidate sets and relevance labels exist.
- Production `matchScore` and `matchLevel` semantics remain blocked until calibration evidence exists.
- Backend/API response validation must enforce candidate membership, uniqueness, score range, and response caps before release.